In [ ]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh
import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import T_inv_sweep, FEMMatrices
from jax_sn.domain.geometric_domain import CrossSectionData

onp.set_printoptions(linewidth = 200)

In [ ]:
#flux_surface     =  jax_sbgeom.flux_surfaces.FluxSurface.from_hdf5("/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4")
#s_mesh           = jnp.linspace(0, 1, 10)
#tetra_mesh       =  jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_tetrahedra(flux_surface, s_mesh, ToroidalExtent.half_module(flux_surface), 50,60)
#face_template    = jsn.tools.mesh_tools._face_template(tetra_mesh[1].shape[1])
#geometric_domain = jsn.domain.GeometricDomain.from_mesh(tetra_mesh, face_template)
#quadrature_set   = jsn.quadrature_set.tn.create_tn_quadrature_set(4)
#pol_mesh         = jax_sbgeom.flux_surfaces.flux_surface_meshing.mesh_poloidal_plane(flux_surface, jnp.linspace(0,1,10)**2, 0.52,  20)
#tri_face_template = jsn.tools.mesh_tools._face_template(pol_mesh[1].shape[1])
#pol_domain        = jsn.domain.GeometricDomain.from_mesh(pol_mesh, tri_face_template)


In [ ]:
from finite_elements.test_finite_elements import _get_element_jsn

In [ ]:

from basix import ElementFamily, LagrangeVariant
import basix
from jax_sn.solution_domain.basix_fem import BasixLagrangianSimplex, _create_face_quadrature_on_face, _map_simplex_point_to_different_permutation, _map_face_points_to_reference_cell
from jax_sn.tools.permutation_tools import _obtain_permutation, _indices_to_permutation_number
import jax_sn
degree = 3
lasimp = BasixLagrangianSimplex(1,3)
lasimp.n_vertices_per_face(0)

In [ ]:
from functools import partial
def _switch_element_type(dimension):
    import StellBlanket as SB
    if dimension == 1:
        return SB.Lines.Line_Element
    elif dimension == 2:
        return SB.Triangles.Triangle_Element
    elif dimension == 3:
        return SB.Tetrahedrons.Tetrahedron_Element
    else:
        raise ValueError("Dimension must be 1, 2 or 3.")
    
def _test_domain_from_sb(dimension, degree):
    import StellBlanket as SB
    material_0 = SB.Generic_Elements.Material(SB.Generic_Elements.Constant_Cross_Section(0.673))
    material_1 = SB.Generic_Elements.Material(SB.Generic_Elements.Constant_Cross_Section(2.5))
    region_sizes=  [0.8, 0.2]
    resolutions  = [4,3]
    size_other_dim = 0.754
    resolutions_other_dim = 2
    element = _switch_element_type(dimension)
    cartesian_domain = SB.Generic_Elements.Create_Slab(element, region_sizes, resolutions, [material_0, material_1], size_other_dim, resolutions_other_dim)    
    return cartesian_domain

def _arbitrary_source_values(n_elements):
    return onp.linspace(1.0, 10.0, n_elements)

def tn_order_index():
    return 4, 3

def _angle_function_indexed(tn_order, index):
    import StellBlanket as SB
    quad_set = SB.Generic_Elements.TN_Quadrature_Set(tn_order)
    return quad_set.Directions()[index:index+1,:]
    
def _matfree_nte_from_sb(dimension, degree, tn_order, angle_index):
    import StellBlanket as SB
    domain   = _test_domain_from_sb(dimension, degree)
    quad_set = SB.Generic_Elements.Quadrature_Set(onp.array(_angle_function_indexed(tn_order, angle_index)), onp.array([1.0]))

    energy_set = SB.Generic_Elements.Energy_Set([0,1])
    matfreente = SB.Generic_Elements.Create_RAM_Simulation(quad_set, energy_set, domain, degree, 0)

    source_values = _arbitrary_source_values(domain.connectivity.shape[0])
    sources = [SB.Generic_Elements.Constant_Source(_switch_element_type(dimension), val, energy_set[0]) for val in source_values]
    total_source = SB.Generic_Elements.Per_Element_Source(sources, domain)
    return matfreente, total_source
    
def _verts_conn_from_sb(dimension, degree):
    import StellBlanket as SB
    domain   = _test_domain_from_sb(dimension, degree)
    return jnp.array(domain.vertices), jnp.array(domain.connectivity)

def _source_from_sb(dimension, degree):
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index()) # angle doesn't matter for this
    source_values = _arbitrary_source_values(nte.geometric_domain.connectivity.shape[0])
    n_values = int(nte.solution_domain.total_size / nte.geometric_domain.connectivity.shape[0])
    return jax_sn.source.Source(jnp.stack([source_values] * n_values, axis=-1))

def _cross_sections_from_sb(dimension, degree):
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index()) # angle doesn't matter for this
    cross_sections = jnp.array([nte.geometric_domain[i].Material().Cross_Section().Value_Total(mfreente.energy_set[0]) for i in range(nte.geometric_domain.connectivity.shape[0])])
    return jax_sn.domain.geometric_domain.CrossSectionData(cross_sections)

def _angle_function_tn():
    return _angle_function_indexed(*tn_order_index())[0]

dimension =3
degree =1
#_test_domain_from_sb(3,2)
#print(_verts_conn_from_sb(dimension, degree))

mfreente, source = _matfree_nte_from_sb(dimension = 3, degree= 2,tn_order = 4, angle_index=0)
from finite_elements.test_finite_elements import create_permutation_dimension


def _expected_values_from_sb(dimension, degree):
    import StellBlanket as SB
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index())
    
    nte.simulation.Sweep(nte.energy_set[0], nte.quadrature_set[0], total_source)
    values = nte.global_solution.All_Solution_Values(nte.energy_set[0], nte.quadrature_set[0]).reshape(nte.geometric_domain.connectivity.shape[0], -1   )

    permutation = create_permutation_dimension(dimension, degree)
    
    return values[:, permutation]
    



In [ ]:
from finite_elements.test_transport_operator import check_transport_inverse_sweep_complete_vmap
print(dimension, degree)
dimension = 2
degree = 0
check_transport_inverse_sweep_complete_vmap(dimension, degree, _verts_conn_from_sb, _angle_function_tn, _expected_values_from_sb, _source_from_sb, _cross_sections_from_sb)

In [ ]:
import StellBlanket as SB

def create_sb_sim(deg ):
    materials = SB.Generic_Elements.Material(SB.Generic_Elements.Constant_Cross_Section(0.673))
    cartesian_tet_domain = SB.Tetrahedrons.Tetrahedron_Create_Slab([1.0, 0.2],[2, 3], [materials, materials], 0.67, 2)

    verts = cartesian_tet_domain.vertices
    conn = cartesian_tet_domain.connectivity
    


    quadrature_set = SB.Generic_Elements.TN_Quadrature_Set(4)


    quad_set = SB.Generic_Elements.Quadrature_Set(quadrature_set.Directions()[0:1], quadrature_set.Weights()[0:1])
    energy_set =SB.Generic_Elements.Energy_Set([0,1])


    matfreente = SB.Generic_Elements.Create_RAM_Simulation(quad_set, energy_set, cartesian_tet_domain, deg, 0)

    source = SB.Generic_Elements.Constant_Source(SB.Tetrahedrons.Tetrahedron_Element, 1.0, matfreente.energy_set[0])
    
    return matfreente, source

def sb_sim_to_jsn(sb_sim, deg):
    quad_set = sb_sim.quadrature_set
    quad_set = jsn.quadrature_set.QuadratureSet(jnp.array(quad_set.Directions()), jnp.array(quad_set.Weights()))       
    simplex_deg = BasixLagrangianSimplex(deg, 3)
    
    #simplex_deg.face_template
    verts = jnp.array(sb_sim.geometric_domain.vertices)
    conn  = jnp.array(sb_sim.geometric_domain.connectivity)
    face_template = jnp.array(simplex_deg.face_template)
    domain = jsn.domain.GeometricDomain.from_mesh((verts, conn), face_template)
    toposort = TopologicalSortedAngle.from_domain_angle(domain, quad_set[0][0])    
    femmatrices = FEMMatrices.from_element(simplex_deg)
    precomputed_data = simplex_deg.precompute_geometric_data(domain)
    cross_sections_data = CrossSectionData(jnp.ones(domain.n_elements)* 0.673)
    source              = jsn.source.Source(jnp.ones((domain.n_elements, simplex_deg.n_basis)))
    sweep_array = jnp.zeros((domain.n_elements, simplex_deg.n_basis))
    
    
    return domain, toposort, femmatrices, precomputed_data, cross_sections_data, source, sweep_array, quad_set

deg = 2
sb_sim, ext_source = create_sb_sim(deg)


In [ ]:
print(sb_sim.geometric_domain.vertices)
sb_sim.geometric_domain.connectivity

In [ ]:
from finite_elements.test_finite_elements import _get_element_jsn, create_permutation, _get_element_old
element = "tetrahedron"
element_old = _get_element_old(element, degree=deg)
permutation = create_permutation(element, element_old, _get_element_jsn(element, degree=deg))
p_inv  = [0] * len(permutation)
for i,v in enumerate(permutation):
    p_inv[v] = i
p_inv

In [ ]:
domain, toposort, femmatrices, precomputed_data, cross_sections_data, source, sweep_array, quad_set = sb_sim_to_jsn(sb_sim, deg)


In [ ]:
sb_sim.simulation.Sweep(sb_sim.energy_set[0], sb_sim.quadrature_set[0], ext_source)

In [ ]:
from jax_sn.tools.permutation_tools import _number_to_permutation

In [ ]:
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_complete_vmap
sweep_result =  transport_inverse_sweep_complete_vmap(femmatrices, domain, toposort, cross_sections_data, precomputed_data, source, sweep_array)

In [ ]:
onp.testing.assert_allclose(sweep_result[:, p_inv],sb_sim.global_solution.All_Solution_Values(sb_sim.energy_set[0], sb_sim.quadrature_set[0]).reshape(sweep_result.shape) )